
# Data Preparation of the Heart Diagnoses file

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))
from src.utils import data_understanding as du





heart_diagnoses_df = pd.read_csv("../../data/raw/heart_diagnoses_1.csv")
heart_diagnoses_df['subject_id'].nunique()

## Data Cleaning

In [ ]:
heart_diagnoses_df



In [ ]:
import re

def detect_gender_regex(text):
    if pd.isna(text):
        return 'unknown'
    
    text_lower = text.lower()

    text_lower = text.lower()

    if match := re.search(r"\b(?:sex|gender|patient sex)\b[^a-z]*\b(m|male)\b", text_lower):
        return "M"
    if match := re.search(r"\b(?:sex|gender|patient sex)\b[^a-z]*\b(f|female)\b", text_lower):
        return "F"

    if re.search(r"\b(mr\.?|mister)\b", text_lower):
        return "M"
    if re.search(r"\b(mrs?\.?|miss|ms\.?)\b", text_lower):
        return "F"
    if re.search(r"\b(he|his|him|himself)\b", text_lower):
        return "M"
    if re.search(r"\b(she|her|hers|herself)\b", text_lower):
        return "F"
    if re.search(r"\b(man|men|male|boy|gentleman)\b", text_lower):
        return "M"
    if re.search(r"\b(woman|women|female|girl|lady)\b", text_lower):
        return "F"

    return np.nan

heart_diagnoses_df['gender'] = heart_diagnoses_df['HPI'].apply(detect_gender_regex)

grouped = heart_diagnoses_df[['subject_id', 'hadm_id', 'gender']].groupby(['subject_id', 'hadm_id'])
# Print the subject the admsission and gender
for (subject_id, hadm_id), group in grouped:
    print(f"Subject ID: {subject_id}, HADM ID: {hadm_id}, Genders: {group['gender'].tolist()}") 



In [ ]:
#drop columns with information not relevant for the analysis: free text columns, reports columns,unusefull columns
heart_diagnoses_df = heart_diagnoses_df.drop(columns=["CT", "MRI", "X-ray", "Ultrasound", "ECG", "CATH", "invasions", "reports","chief_complaint", "physical_exam","HPI","anchor_year","note_seq","note_type","charttime","storetime"])
heart_diagnoses_df_columns= du.get_cols(heart_diagnoses_df)
print("Columns in heart diagnoses dataset:",heart_diagnoses_df_columns) 


In [ ]:
#Missing value analysis
#age and gender have missing values but they can be usfull for the analysis so we will keep them
#dod has many missing values can be  that the patient witout dod is alive or the data is missing, so is ambiguous, we will drop it
heart_diagnoses_df=heart_diagnoses_df.drop(columns=["dod"])
heart_diagnoses_df['subject_id'].nunique()


In [ ]:
# Calcola il numero di occorrenze per ogni icd_code
icd_counts = heart_diagnoses_df['icd_code'].value_counts()

# Calcola Q1, Q3 e IQR
q1 = icd_counts.quantile(0.25)
q3 = icd_counts.quantile(0.75)
iqr = q3 - q1

# Calcola i limiti
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

print(f"Q1 (25%): {q1}")
print(f"Q3 (75%): {q3}")
print(f"IQR: {iqr}")
print(f"Lower limit: {lower_limit}")
print(f"Upper limit: {upper_limit}")

# Identifica gli icd_code da rimuovere (sotto il limite inferiore)
rare_icd_codes_iqr = icd_counts[icd_counts < lower_limit].index

print(f"\nICD codes da rimuovere (sotto il limite): {len(rare_icd_codes_iqr)}")
print(rare_icd_codes_iqr.tolist())

# Rimuovi gli icd_code rari
heart_diagnoses_df = heart_diagnoses_df[~heart_diagnoses_df['icd_code'].isin(rare_icd_codes_iqr)]

print(f"\nShape dopo rimozione: {heart_diagnoses_df.shape}")
print(f"ICD codes rimasti: {heart_diagnoses_df['icd_code'].nunique()}")
icd_counts_after = heart_diagnoses_df['icd_code'].value_counts()
icd_counts_after

In [ ]:
# Step 1: Raggruppa per paziente e controlla se ha almeno un age noto
patients_with_age_info = (
    heart_diagnoses_df.groupby("subject_id")["age"]
    .apply(lambda x: x.notna().any())   # True se almeno un record ha gender
)

# Step 2: Filtra i pazienti senza nessuna info sul sesso
patients_without_age_info = patients_with_age_info[~patients_with_age_info]

# Step 3: Conta quanti sono
num_patients_without_age_info = patients_without_age_info.shape[0]

print(f"✓ Numero di pazienti senza nessuna info sul sesso: {num_patients_without_age_info}")

In [ ]:
# Caso 1: Pazienti con una sola ammissione → imputazione con mediana ICD

admissions_per_patient = (
    heart_diagnoses_df.groupby('subject_id')['hadm_id']
    .nunique()
    .reset_index()
)
admissions_per_patient.columns = ['subject_id', 'num_admissions']
single_admission_patients = admissions_per_patient[
    admissions_per_patient['num_admissions'] == 1
]['subject_id'].tolist()
median_age_per_icd = (
    heart_diagnoses_df[heart_diagnoses_df['age'].notna()]
    .groupby('icd_code')['age']
    .median()
    .astype(int)
    .reset_index()
)
median_age_per_icd.columns = ['icd_code', 'median_age']

mask_single_admission = heart_diagnoses_df['subject_id'].isin(single_admission_patients)
mask_missing_age = heart_diagnoses_df['age'].isna()
mask_has_median = heart_diagnoses_df['icd_code'].isin(median_age_per_icd['icd_code'])

rows_to_fill_single = heart_diagnoses_df[mask_single_admission & mask_missing_age & mask_has_median]

for idx in rows_to_fill_single.index:
    icd_code = heart_diagnoses_df.loc[idx, 'icd_code']
    median_age = median_age_per_icd[median_age_per_icd['icd_code'] == icd_code]['median_age'].values
    if len(median_age) > 0:
        heart_diagnoses_df.loc[idx, 'age'] = median_age[0]

print(f"✓ Età imputata con mediana ICD per {rows_to_fill_single.shape[0]} record (single admission)")


In [ ]:
# Caso 2: Pazienti con più ammissioni e almeno un'età osservata → usa direttamente quell'età
multi_admission_patients = admissions_per_patient[admissions_per_patient['num_admissions'] > 1]['subject_id'].tolist()

observed_age_per_patient = (
    heart_diagnoses_df[heart_diagnoses_df['subject_id'].isin(multi_admission_patients) & heart_diagnoses_df['age'].notna()]
    .groupby('subject_id')['age']
    .first()   # usa direttamente la prima età trovata
    .reset_index()
    .rename(columns={'age': 'patient_observed_age'})
)

# Merge nel dataset
heart_diagnoses_df = heart_diagnoses_df.merge(observed_age_per_patient, on='subject_id', how='left')

mask_multi_admission = heart_diagnoses_df['subject_id'].isin(multi_admission_patients)
mask_missing_age = heart_diagnoses_df['age'].isna()
mask_has_observed = heart_diagnoses_df['patient_observed_age'].notna()

rows_to_fill_observed = heart_diagnoses_df[mask_multi_admission & mask_missing_age & mask_has_observed]

for idx in rows_to_fill_observed.index:
    heart_diagnoses_df.loc[idx, 'age'] = heart_diagnoses_df.loc[idx, 'patient_observed_age']

print(f"✓ Età imputata con valore osservato per {rows_to_fill_observed.shape[0]} record (multi admission con info)")
heart_diagnoses_df = heart_diagnoses_df.drop(columns=['patient_observed_age'])


In [ ]:
# Caso 3: Pazienti con più ammissioni e nessuna età osservata → imputazione con mediana delle mediane ICD
df_with_median = heart_diagnoses_df.merge(median_age_per_icd, on='icd_code', how='left')

median_of_medians_per_patient = (
    df_with_median[df_with_median['subject_id'].isin(multi_admission_patients)]
    .groupby('subject_id')['median_age']
    .median()
    .reset_index()
    .rename(columns={'median_age': 'patient_imputed_age'})
)

# Merge nel dataset
heart_diagnoses_df = heart_diagnoses_df.merge(median_of_medians_per_patient, on='subject_id', how='left')

mask_multi_admission = heart_diagnoses_df['subject_id'].isin(multi_admission_patients)
mask_missing_age = heart_diagnoses_df['age'].isna()
mask_has_imputed = heart_diagnoses_df['patient_imputed_age'].notna()

rows_to_fill_imputed = heart_diagnoses_df[mask_multi_admission & mask_missing_age & mask_has_imputed]

for idx in rows_to_fill_imputed.index:
    heart_diagnoses_df.loc[idx, 'age'] = heart_diagnoses_df.loc[idx, 'patient_imputed_age']

print(f"✓ Età imputata con mediana delle mediane ICD per {rows_to_fill_imputed.shape[0]} record (multi admission senza info)")
heart_diagnoses_df = heart_diagnoses_df.drop(columns=['patient_imputed_age'])



In [ ]:
#print the number of patient without age information

# 1. Trova i soggetti che hanno TUTTI i record con age mancante
subjects_missing_age = (
    heart_diagnoses_df.groupby("subject_id")["age"]
    .apply(lambda x: x.isna().all())
)

# 2. Lista dei soggetti senza età
subjects_missing_age_list = subjects_missing_age[subjects_missing_age].index.tolist()

# 3. Filtra il dataset per questi soggetti
df_missing_age = heart_diagnoses_df[heart_diagnoses_df["subject_id"].isin(subjects_missing_age_list)]

# 4. Raggruppa e stampa gli ICD code per ciascun soggetto
icd_per_missing_subject = (
    df_missing_age.groupby("subject_id")["icd_code"]
    .apply(lambda x: sorted(x.dropna().unique().tolist()))
    .reset_index(name="icd_codes")
)

print(icd_per_missing_subject)






In [ ]:
#per questi in base al web mettere come età 70 anni
# Step 5: Sostituisci l'età mancante con 70 per i soggetti senza alcuna info
mode_age=heart_diagnoses_df['age'].mode()[0]
mask_missing_subjects = heart_diagnoses_df["subject_id"].isin(subjects_missing_age_list) & heart_diagnoses_df["age"].isna()

heart_diagnoses_df.loc[mask_missing_subjects, "age"] = mode_age

# Log finale
print(f"✓ Età imputata con la moda dell'età per {mask_missing_subjects.sum()} record")
print(heart_diagnoses_df[heart_diagnoses_df['subject_id'].isin(subjects_missing_age_list)][['subject_id','icd_code','age']].head(20))




In [ ]:
#distributiom of the age
sns.histplot(heart_diagnoses_df, x="age")

In [ ]:
# Step 1: Raggruppa per paziente e controlla se ha almeno un gender noto
patients_with_gender_info = (
    heart_diagnoses_df.groupby("subject_id")["gender"]
    .apply(lambda x: x.notna().any())   # True se almeno un record ha gender
)

# Step 2: Filtra i pazienti senza nessuna info sul sesso
patients_without_gender_info = patients_with_gender_info[~patients_with_gender_info]

# Step 3: Conta quanti sono
num_patients_without_gender_info = patients_without_gender_info.shape[0]

print(f"✓ Numero di pazienti senza nessuna info sul sesso: {num_patients_without_gender_info}")



In [ ]:
#metto il gender per chi ha solo un hd mantenendo la proporzione
np.random.seed(42)

def fill_with_proportions(group, global_props=None):
    missing_mask = group['gender'].isna()
    if missing_mask.sum() == 0:
        return group
    
    # Conta valori disponibili
    value_counts = group['gender'].value_counts()
    
    if value_counts.empty:
        # Se non ci sono valori osservati per quell'ICD
        if global_props is not None:
            # Usa proporzioni globali
            choices = np.random.choice(global_props.index, size=missing_mask.sum(), p=global_props.values)
            group.loc[missing_mask, 'gender'] = choices
        else:
            # Altrimenti assegna un default (es. 'M')
            group.loc[missing_mask, 'gender'] = 'M'
    else:
        # Usa proporzioni locali ICD
        proportions = value_counts / value_counts.sum()
        choices = np.random.choice(value_counts.index, size=missing_mask.sum(), p=proportions.values)
        group.loc[missing_mask, 'gender'] = choices
    
    return group

# Calcola proporzioni globali F/M
global_props = heart_diagnoses_df['gender'].value_counts(normalize=True)

# Applica solo ai pazienti con una sola ammissione
df_single_admission = heart_diagnoses_df[heart_diagnoses_df['subject_id'].isin(single_admission_patients)]
df_single_admission = df_single_admission.groupby('icd_code', group_keys=False).apply(fill_with_proportions, global_props=global_props)

# Aggiorna dataset originale
heart_diagnoses_df.update(df_single_admission)

print(f"✓ Missing values in gender (single admission): {df_single_admission['gender'].isna().sum()}")
print(df_single_admission['gender'].value_counts())

In [ ]:
# Fill missing gender using other admissions of the same subject
multi_admission_subjects = (
    heart_diagnoses_df.groupby('subject_id')['hadm_id'].nunique().gt(1)
)

gender_lookup = (
    heart_diagnoses_df[heart_diagnoses_df['gender'].notna()]
    .sort_values('hadm_id')
    .drop_duplicates('subject_id', keep='first')
    .set_index('subject_id')['gender']
)

mask_missing_gender = heart_diagnoses_df['gender'].isna()
mask_multi_with_gender = (
    multi_admission_subjects.reindex(heart_diagnoses_df['subject_id']).fillna(False).values &
    heart_diagnoses_df['subject_id'].isin(gender_lookup.index)
)

heart_diagnoses_df.loc[mask_missing_gender & mask_multi_with_gender, 'gender'] = (
    heart_diagnoses_df.loc[mask_missing_gender & mask_multi_with_gender, 'subject_id']
    .map(gender_lookup)
)

print(f"✓ Missing values in gender (multi admission): {heart_diagnoses_df['gender'].isna().sum()}")

In [ ]:
#Step 1: Raggruppa per paziente e controlla se ha almeno un gender noto
patients_with_gender_info = (
    heart_diagnoses_df.groupby("subject_id")["gender"]
    .apply(lambda x: x.notna().any())   # True se almeno un record ha gender
)

# Step 2: Filtra i pazienti senza nessuna info sul sesso
patients_without_gender_info = patients_with_gender_info[~patients_with_gender_info]

# Step 3: Conta quanti sono
num_patients_without_gender_info = patients_without_gender_info.shape[0]

print(f"✓ Numero di pazienti senza nessuna info sul sesso: {num_patients_without_gender_info}")


In [ ]:
#to visualize the number of female and men for each icd_code
sns.catplot(data=heart_diagnoses_df, x="icd_code", hue="gender",kind="count")

In [ ]:
#drop columns that have info achieved in another coloumns 
heart_diagnoses_df=heart_diagnoses_df.drop(columns=["note_id","subject_id_dx"])


In [ ]:
#final dataset analysis
print("Shape of heart diagnoses dataset:",)
heart_diagnoses_df.shape
print("Info of heart diagnoses dataset:")
heart_diagnoses_df.info()
heart_diagnoses_df_none=du.get_cols_with_none(heart_diagnoses_df)
print("Missing values in each column of heart diagnoses dataset:\n",heart_diagnoses_df_none)
print("Basic statistics of heart diagnoses dataset:")
du.get_basic_stats(heart_diagnoses_df)


In [ ]:
# mean and median age for each icd_code
icd_stats = heart_diagnoses_df.groupby('icd_code')['age'].agg(['mean', 'median']).reset_index()
icd_stats.columns = ['icd_code', 'mean_age', 'median_age']
print("Mean and median age for each ICD code:\n", icd_stats)

In [ ]:
# visualize the new icd_code distribution
sns.histplot(heart_diagnoses_df, x="icd_code")


In [ ]:
# visualize the number of female and men for each icd_code
sns.catplot(data=heart_diagnoses_df, x="icd_code", hue="gender",kind="count")

In [ ]:
# visualize for each icd_code the age of the patient 
sns.boxplot(data=heart_diagnoses_df,x="icd_code",y="age")

In [ ]:

heart_diagnoses_df.to_csv("../../data/processed/hd.csv", index=False)

In [ ]:
heart_diagnoses_df